<div>

# **AI agent use Tavily**

<div dir=RTL>

# **התקנות / ספריות**

In [2]:
# התקנה
!pip install -q langgraph langchain-google-genai langchain-community tavily-python
# ספריות
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults

TimeoutException: Requesting secret TAVILY_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

<div dir=RTL>

# **הגדרת מודל וכלים**

In [ ]:
# מפתחות
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
# כלים
tools = [TavilySearchResults(max_results=3)]

# מודל
model = ChatGoogleGenerativeAI(
    model="gemini-pro-latest",
    google_api_key=os.environ["GOOGLE_API_KEY"]
).bind_tools(tools)

<div dir=RTL>

# **הגדרת גרף**

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# צומת המודל
def call_model(state: MessagesState):
    return {"messages": [model.invoke(state["messages"])]}

# בדיקה: האם המודל ביקש להפעיל כלי?
def should_continue(state: MessagesState):
    return "tools" if state["messages"][-1].tool_calls else END

# בניית הגרף
graph = StateGraph(MessagesState)
graph.add_node("model", call_model)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "model")
graph.add_conditional_edges("model", should_continue)
graph.add_edge("tools", "model")

agent = graph.compile()

# הצגת הגרף
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

<div dir=RTL>

# **הפעלת סוכן**

In [ ]:
from langchain_core.messages import HumanMessage

# שאל את הסוכן שאלה
question = "מה החדשות הכי מעניינות בתחום ה-AI היום?"
result = agent.invoke({"messages": [HumanMessage(content=question)]})

<div dir=RTL>

# **הדפסת התשובה**

In [ ]:
# הדפסת תשובה
print(result["messages"][-1].content)